# Activité Pratique N°4 — Architecture Transformer
## Partie 1 — Étape 5 : Modèle de type GPT

**IA Générative Multi Modale — 5IAD G1, EMSI** · Étudiant : BENHAYOUN SADAFI Mohamed

Créer et tester un exemple basé sur un Transformer de type GPT : prédire la suite d'un texte.

GPT-2 est un **décodeur Transformer à attention causale** : chaque token ne voit que les tokens précédents.
Il génère **un token à la fois** : il calcule les probabilités du token suivant, en choisit un, l'ajoute à la séquence et recommence.

```
Input : séquence « Artificial Intelligence can… » ──► Generative Pre-Trained Transformer ──► Output : « …help people and organizations… »
```

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
modele = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
tokenizer.pad_token = tokenizer.eos_token          # GPT-2 n'a pas de token de remplissage
modele.config.pad_token_id = tokenizer.pad_token_id
print("Paramètres :", sum(p.numel() for p in modele.parameters()) / 1e6, "millions")

C:\Users\user\Desktop\5IAD G1 BENHAYOUN SADAFI Mohamed, emsi, matiere ia multimodale\Activité Pratique N°4 - Architecture Transformer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\user\Desktop\5IAD G1 BENHAYOUN SADAFI Mohamed, emsi, matiere ia multimodale\Activité Pratique N°4 - Architecture Transformer\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


C:\Users\user\Desktop\5IAD G1 BENHAYOUN SADAFI Mohamed, emsi, matiere ia multimodale\Activité Pratique N°4 - Architecture Transformer\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   7%|▋         | 10/148 [00:00<00:01, 92.44it/s]

Loading weights:  14%|█▎        | 20/148 [00:00<00:02, 44.66it/s]

Loading weights:  26%|██▌       | 38/148 [00:00<00:01, 78.27it/s]

Loading weights:  33%|███▎      | 49/148 [00:01<00:04, 23.88it/s]

Loading weights: 100%|██████████| 148/148 [00:01<00:00, 90.91it/s]

Paramètres : 124.439808 millions


### 1. Tokenisation et prédiction d'un seul token suivant

In [2]:
amorce = "Artificial intelligence can"
entrees = tokenizer(amorce, return_tensors="pt").to(device)
print("Tokens :", tokenizer.convert_ids_to_tokens(entrees["input_ids"][0]))
print("Dimensions de l'entrée :", tuple(entrees["input_ids"].shape))

with torch.no_grad():
    logits = modele(**entrees).logits          # (1, nb_tokens, taille_vocabulaire)
print("Logits :", tuple(logits.shape))

probas = torch.softmax(logits[0, -1], dim=-1)  # distribution du token suivant
top = torch.topk(probas, 5)
print(f"\nToken suivant le plus probable après {amorce!r} :")
for p, i in zip(top.values, top.indices):
    print(f"  {p.item():6.2%}  {tokenizer.decode(i)!r}")

Tokens : ['Art', 'ificial', 'Ġintelligence', 'Ġcan']
Dimensions de l'entrée : (1, 4)


Logits : (1, 4, 50257)



Token suivant le plus probable après 'Artificial intelligence can' :
  15.29%  ' be'
   7.19%  ' also'
   7.05%  ' help'
   2.80%  "'t"
   2.52%  ' do'


### 2. Générer la suite du texte, token après token

In [3]:
torch.manual_seed(0)
with torch.no_grad():
    sorties = modele.generate(**entrees, max_new_tokens=50, do_sample=True, temperature=0.8, top_p=0.9,
                              num_return_sequences=3, pad_token_id=tokenizer.pad_token_id)

for k, sequence in enumerate(sorties, 1):
    print(f"--- Suite {k} ---")
    print(tokenizer.decode(sequence, skip_special_tokens=True), "\n")

--- Suite 1 ---
Artificial intelligence can't do anything to fix the problem. It doesn't help the people who need it the most. It does nothing.

The problem with artificial intelligence is that it's been designed to do nothing. The problem is that it doesn't work. 

--- Suite 2 ---
Artificial intelligence can help us better understand our world, and help us better understand our world's challenges.

Machine learning can help us better understand our world, and help us better understand our world's challenges.

Machine learning can help us better understand our world 

--- Suite 3 ---
Artificial intelligence can play an important role in helping us better understand our world and to better understand our everyday lives.

But some people's experience with AI is less clear-cut.

For instance, some people are more interested in helping people than in helping 



- `do_sample=True` : on tire le token au hasard selon les probabilités (sinon : toujours le plus probable, texte répétitif).
- `temperature=0.8` : distribution un peu plus pointue que l'originale.
- `top_p=0.9` : on ne garde que les tokens les plus probables dont la somme des probabilités atteint 90 %.

**À retenir :** GPT prédit le prochain token à partir du passé. Il produit une suite **plausible**, sans garantie qu'elle soit vraie.